# Traffic Count Data Aggregator

The **Traffic Count Data** consists of **per-vehicle observations collected from approximately 300 traffic sensors across Ireland**. The data is made available as **daily datasets**, where each file contains records of individual vehicles detected by these sensors.

Each day's dataset can be accessed using the following URL format:

**https://data.tii.ie/Datasets/TrafficCountData/yyyy/MM/dd/per-vehicle-records-yyyy-MM-dd.csv**

By replacing `yyyy`, `MM`, and `dd` with the corresponding **year, month, and day**, the dataset for a specific date can be retrieved.

Since the data is recorded **per vehicle**, even a single day's dataset can be **very large**, containing detailed information about each vehicle passing through the sensors.

This notebook focuses on **collecting the per-vehicle datasets for all days between 2019 and 2020** and **aggregating them to build a primary dataset**, producing **hourly traffic statistics per sensor and lane** for further analysis and modeling.

## Imports

In [1]:
# Core libraries
import os
import requests
from datetime import date, timedelta

# Progress tracking
from tqdm import tqdm

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, max, min, stddev,
    count, when, countDistinct
)

## Spark Session Initialization

In [2]:
spark = SparkSession.builder.appName("TrafficDataAggregator").getOrCreate()

## Method to download per-vehicle dataset for a date

In [3]:
def download_traffic_data(target_date):
    
    # Ensure temp folder exists
    os.makedirs("temp", exist_ok=True)
    
    # Format date parts
    yyyy = target_date.year
    mm = f"{target_date.month:02d}"
    dd = f"{target_date.day:02d}"
    
    # Build URL
    url = f"https://data.tii.ie/Datasets/TrafficCountData/{yyyy}/{mm}/{dd}/per-vehicle-records-{yyyy}-{mm}-{dd}.csv"
    
    # Local file path
    file_path = f"temp/per-vehicle-records-{yyyy}-{mm}-{dd}.csv"

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:
            with open(file_path, "wb") as f:
                f.write(response.content)

            print(f"Downloaded: {file_path}")
            return file_path

        else:
            print(f"No dataset available for {target_date}")
            return None

    except Exception as e:
        print(f"Error downloading {target_date}: {e}")
        return None

## Method to Aggregate per-vehicle dataset to per-hour

In [4]:
def aggregate_daily_dataset(file_path):

    # Read CSV
    df = spark.read.option("header", True).option("inferSchema", True).csv(file_path)

    # Grouping columns
    group_cols = ["cosit", "lane", "year", "month", "day", "hour"]

    # Aggregate traffic statistics
    df_hour = df.groupBy(group_cols).agg(

        count("*").alias("vehicle_count"),

        avg("speed").alias("avg_speed"),
        max("speed").alias("max_speed"),
        min("speed").alias("min_speed"),
        stddev("speed").alias("speed_std"),

        avg("length").alias("avg_length"),
        max("length").alias("max_length"),
        stddev("length").alias("length_std"),

        avg("headway").alias("avg_headway"),
        max("headway").alias("max_headway"),
        min("headway").alias("min_headway"),
        stddev("headway").alias("headway_std"),

        avg("gap").alias("avg_gap"),
        max("gap").alias("max_gap"),
        min("gap").alias("min_gap"),
        stddev("gap").alias("gap_std"),

        count(when(col("classname") == "CAR", True)).alias("CAR_count"),
        count(when(col("classname") == "HGV_ART", True)).alias("HGV_ART_count"),
        count(when(col("classname") == "BUS", True)).alias("BUS_count"),
        count(when(col("classname") == "HGV_RIG", True)).alias("HGV_RIG_count"),
        count(when(col("classname") == "CARAVAN", True)).alias("CARAVAN_count"),
        count(when(col("classname") == "LGV", True)).alias("LGV_count"),
        count(when(col("classname") == "MBIKE", True)).alias("MBIKE_count")

    )

    return df_hour

## Iterate over the date range and aggregate the dataset

In [6]:
os.makedirs("datasets", exist_ok=True)

start_date = date(2019, 1, 1)
end_date = date(2019, 12, 31)

current = start_date

current_quarter = None
current_year = None
quarter_file = None

while current <= end_date:
    print(f"Processing {current}")

    # Determine quarter
    q = (current.month - 1) // 3 + 1
    y = current.year
    quarter_file = f"datasets/{y}_Q{q}.csv"

    # Download
    file_path = download_traffic_data(current)

    if file_path and os.path.exists(file_path):
        # Aggregate
        df_hour = aggregate_daily_dataset(file_path).toPandas()

        # Append to existing quarter file or create new
        if os.path.exists(quarter_file):
            # Incremental append
            df_hour.to_csv(quarter_file, mode="a", header=False, index=False)
        else:
            # First day of the quarter -- write with header
            df_hour.to_csv(quarter_file, mode="w", header=True, index=False)

        # Delete raw CSV
        os.remove(file_path)

        print(f"Updated {quarter_file} with {current}")

    else:
        print(f"Data not available for {current}, skipping.")

    current += timedelta(days=1)

Processing 2019-01-01
Downloaded: temp/per-vehicle-records-2019-01-01.csv
Updated datasets/2019_Q1.csv with 2019-01-01
Processing 2019-01-02
Downloaded: temp/per-vehicle-records-2019-01-02.csv
Updated datasets/2019_Q1.csv with 2019-01-02
Processing 2019-01-03
Downloaded: temp/per-vehicle-records-2019-01-03.csv
Updated datasets/2019_Q1.csv with 2019-01-03
Processing 2019-01-04
Downloaded: temp/per-vehicle-records-2019-01-04.csv
Updated datasets/2019_Q1.csv with 2019-01-04
Processing 2019-01-05
Downloaded: temp/per-vehicle-records-2019-01-05.csv
Updated datasets/2019_Q1.csv with 2019-01-05
Processing 2019-01-06
Downloaded: temp/per-vehicle-records-2019-01-06.csv
Updated datasets/2019_Q1.csv with 2019-01-06
Processing 2019-01-07
Downloaded: temp/per-vehicle-records-2019-01-07.csv
Updated datasets/2019_Q1.csv with 2019-01-07
Processing 2019-01-08
Downloaded: temp/per-vehicle-records-2019-01-08.csv
Updated datasets/2019_Q1.csv with 2019-01-08
Processing 2019-01-09
Downloaded: temp/per-vehic

In [ ]:
55